# Cell Counting in Microscopy Images with U-Net

This notebook trains a **U-Net** convolutional neural network to segment and count cells in 128x128 grayscale microscopy images.

**U-Net** is an encoder-decoder architecture originally designed for biomedical image segmentation ([Ronneberger et al., 2015](https://arxiv.org/abs/1505.04597)). The encoder progressively downsamples the input to capture context, while the decoder upsamples back to the original resolution. Skip connections between corresponding encoder and decoder layers preserve fine-grained spatial detail that would otherwise be lost during downsampling.

**Task:** Given a microscopy image, predict a binary segmentation mask (cell vs. background), then count individual cells using connected-component analysis.

## Setup & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import cv2

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 0.0003
NUM_EPOCHS = 65
TRAIN_SPLIT = 0.8

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Data Loading

In [ ]:
data_labeled = np.load('data/train_data.npz')
test_images = np.load('data/test_images.npz')

X_labeled = data_labeled['X']
y_labeled = data_labeled['y']

X_train, X_val, y_train, y_val = train_test_split(
    X_labeled, y_labeled, train_size=TRAIN_SPLIT, random_state=42
)

print(f'Training set:   {X_train.shape[0]} images')
print(f'Validation set: {X_val.shape[0]} images')
print(f'Test set:       {test_images["X"].shape[0]} images')
print(f'Image shape:    {X_train.shape[1:]}')

## Exploratory Visualization

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col in range(4):
    i = np.random.randint(len(X_train))
    axes[0, col].imshow(X_train[i], cmap='gray')
    axes[0, col].set_title('Input')
    axes[0, col].axis('off')
    axes[1, col].imshow(y_train[i], cmap='gray')
    axes[1, col].set_title('Mask')
    axes[1, col].axis('off')
plt.suptitle('Sample Training Images and Segmentation Masks')
plt.tight_layout()
plt.show()

## Dataset & DataLoader

In [ ]:
class CellsDataset:
    """Dataset for paired microscopy images and segmentation masks."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i] / 255.0
        y = self.y[i]
        x = x.reshape((1, 128, 128))
        y = y.reshape((1, 128, 128))
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        return x, y


class CellsDatasetTest:
    """Dataset for unlabeled test microscopy images."""

    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i] / 255.0
        x = x.reshape((1, 128, 128))
        x = torch.tensor(x, dtype=torch.float32)
        return x


dataset_train = CellsDataset(X_train, y_train)
dataset_val = CellsDataset(X_val, y_val)

dataloader_train = torch.utils.data.DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
dataloader_val = torch.utils.data.DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)

## Model Architecture

The U-Net has four encoder stages (each: two 3x3 convolutions + ReLU, then 2x2 max-pool) that progressively extract features at 64, 128, 256, and 512 channels. The bottleneck operates at 1024 channels. Four decoder stages mirror the encoder, each upsampling and concatenating the corresponding encoder features via skip connections before applying convolutions. A final 1x1-equivalent convolution produces a single-channel segmentation map.

In [ ]:
class UNet(torch.nn.Module):
    """U-Net for 128x128 grayscale image segmentation.

    Encoder:  1 -> 64 -> 128 -> 256 -> 512 channels (with max-pooling)
    Bottleneck: 512 -> 1024 channels
    Decoder:  1024 -> 512 -> 256 -> 128 -> 64 channels (with upsampling + skip connections)
    Output:   64 -> 1 channel (logits)
    """

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1_conv1 = torch.nn.Conv2d(1, 64, kernel_size=3, padding='same')
        self.enc1_conv2 = torch.nn.Conv2d(64, 64, kernel_size=3, padding='same')

        self.enc2_conv1 = torch.nn.Conv2d(64, 128, kernel_size=3, padding='same')
        self.enc2_conv2 = torch.nn.Conv2d(128, 128, kernel_size=3, padding='same')

        self.enc3_conv1 = torch.nn.Conv2d(128, 256, kernel_size=3, padding='same')
        self.enc3_conv2 = torch.nn.Conv2d(256, 256, kernel_size=3, padding='same')

        self.enc4_conv1 = torch.nn.Conv2d(256, 512, kernel_size=3, padding='same')
        self.enc4_conv2 = torch.nn.Conv2d(512, 512, kernel_size=3, padding='same')

        # Bottleneck
        self.bottleneck_conv1 = torch.nn.Conv2d(512, 1024, kernel_size=3, padding='same')
        self.bottleneck_conv2 = torch.nn.Conv2d(1024, 1024, kernel_size=3, padding='same')

        # Decoder
        self.up4_conv = torch.nn.Conv2d(1024, 512, kernel_size=3, padding='same')
        self.dec4_conv1 = torch.nn.Conv2d(1024, 512, kernel_size=3, padding='same')
        self.dec4_conv2 = torch.nn.Conv2d(512, 512, kernel_size=3, padding='same')

        self.up3_conv = torch.nn.Conv2d(512, 256, kernel_size=3, padding='same')
        self.dec3_conv1 = torch.nn.Conv2d(512, 256, kernel_size=3, padding='same')
        self.dec3_conv2 = torch.nn.Conv2d(256, 256, kernel_size=3, padding='same')

        self.up2_conv = torch.nn.Conv2d(256, 128, kernel_size=3, padding='same')
        self.dec2_conv1 = torch.nn.Conv2d(256, 128, kernel_size=3, padding='same')
        self.dec2_conv2 = torch.nn.Conv2d(128, 128, kernel_size=3, padding='same')

        self.up1_conv = torch.nn.Conv2d(128, 64, kernel_size=3, padding='same')
        self.dec1_conv1 = torch.nn.Conv2d(128, 64, kernel_size=3, padding='same')
        self.dec1_conv2 = torch.nn.Conv2d(64, 64, kernel_size=3, padding='same')

        self.final_conv = torch.nn.Conv2d(64, 1, kernel_size=3, padding='same')

        # Shared operations
        self.relu = torch.nn.ReLU()
        self.maxpool = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        self.upsample = torch.nn.Upsample(scale_factor=2)

    def forward(self, x):
        # Encoder
        x = self.relu(self.enc1_conv1(x))
        skip1 = self.relu(self.enc1_conv2(x))
        x = self.maxpool(skip1)

        x = self.relu(self.enc2_conv1(x))
        skip2 = self.relu(self.enc2_conv2(x))
        x = self.maxpool(skip2)

        x = self.relu(self.enc3_conv1(x))
        skip3 = self.relu(self.enc3_conv2(x))
        x = self.maxpool(skip3)

        x = self.relu(self.enc4_conv1(x))
        skip4 = self.relu(self.enc4_conv2(x))
        x = self.maxpool(skip4)

        # Bottleneck
        x = self.relu(self.bottleneck_conv1(x))
        x = self.relu(self.bottleneck_conv2(x))

        # Decoder
        x = self.relu(self.up4_conv(self.upsample(x)))
        x = torch.cat((x, skip4), dim=1)
        x = self.relu(self.dec4_conv1(x))
        x = self.relu(self.dec4_conv2(x))

        x = self.relu(self.up3_conv(self.upsample(x)))
        x = torch.cat((x, skip3), dim=1)
        x = self.relu(self.dec3_conv1(x))
        x = self.relu(self.dec3_conv2(x))

        x = self.relu(self.up2_conv(self.upsample(x)))
        x = torch.cat((x, skip2), dim=1)
        x = self.relu(self.dec2_conv1(x))
        x = self.relu(self.dec2_conv2(x))

        x = self.relu(self.up1_conv(self.upsample(x)))
        x = torch.cat((x, skip1), dim=1)
        x = self.relu(self.dec1_conv1(x))
        x = self.relu(self.dec1_conv2(x))

        x = self.final_conv(x)
        return x


model = UNet().to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = torch.nn.BCEWithLogitsLoss()

train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    for x_batch, y_batch in dataloader_train:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(x_batch)
        loss = loss_fn(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        train_loss = 0
        for x_batch, y_batch in dataloader_train:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            train_loss += loss_fn(outputs, y_batch).item() * len(y_batch)
        train_loss /= len(dataset_train)
        train_losses.append(train_loss)

        val_loss = 0
        for x_batch, y_batch in dataloader_val:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            val_loss += loss_fn(outputs, y_batch).item() * len(y_batch)
        val_loss /= len(dataset_val)
        val_losses.append(val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')

## Training Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Training')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Qualitative Evaluation

In [ ]:
sigmoid = torch.nn.Sigmoid()

model.eval()
with torch.no_grad():
    x_batch, y_batch = next(iter(dataloader_val))
    x_batch = x_batch.to(device)
    y_pred = sigmoid(model(x_batch)).cpu().numpy()
    y_batch = y_batch.numpy()
    x_batch = x_batch.cpu().numpy()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for col in range(4):
    i = np.random.randint(len(x_batch))
    axes[0, col].imshow(x_batch[i][0], cmap='gray')
    axes[0, col].set_title('Input')
    axes[0, col].axis('off')

    axes[1, col].imshow(y_batch[i][0], cmap='gray')
    axes[1, col].set_title('Ground Truth')
    axes[1, col].axis('off')

    axes[2, col].imshow(y_pred[i][0], cmap='gray')
    axes[2, col].set_title('Prediction')
    axes[2, col].axis('off')

plt.suptitle('Input / Ground Truth / Predicted Segmentation')
plt.tight_layout()
plt.show()

## Cell Counting on Test Set

For each test image, the model predicts a segmentation mask. The predicted probabilities are thresholded at 0.5, and `cv2.connectedComponents` counts the number of distinct cell regions.

In [ ]:
dataset_test = CellsDatasetTest(test_images['X'])
dataloader_test = torch.utils.data.DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

cell_counts = []

model.eval()
with torch.no_grad():
    for x_batch in dataloader_test:
        x_batch = x_batch.to(device)
        outputs = model(x_batch)
        probabilities = sigmoid(outputs)
        labels = torch.round(probabilities)

        for i in range(len(x_batch)):
            mask = labels[i].cpu().numpy().astype('uint8')[0]
            num_components, _ = cv2.connectedComponents(mask)
            cell_counts.append(num_components)

df_submit = pd.DataFrame({'index': range(len(cell_counts)), 'count': cell_counts})
df_submit.to_csv('cell_counts.csv', index=False)
print(f'Counted cells in {len(cell_counts)} test images')
print(df_submit.head(10))